# Self-query Retriever로 자연어 필터 생성

Self-query Retriever는 사용자의 자연어 질문에서 **검색어와 metadata 조건을 스스로 분리하는 Retriever**이다. 여기서 Self-query는 모델을 다시 학습한다는 뜻이 아니라, LLM이 검색용 구조를 만든다는 뜻이다.

### 자연어 질문에서 분리하는 세 요소

사용자가 `김민수가 작성한 검색 관련 문서 2개만 찾아줘`라고 질문하면 다음 세 가지 검색 조건이 필요하다.

- 검색어: `검색 관련`이다.
- metadata 조건: `author == 김민수`이다.
- 반환 개수: `2개`이다.

Self-query Retriever는 LLM으로 이 조건들을 자동 추출하고 Vector Store가 사용할 검색 구조로 변환한다. 이 과정에서는 **자연어 질문이 실제 검색 조건으로 바뀌는 과정**을 확인한다. 최종 반환값은 답변이 아니라 `list[Document]`이다.

### 세 검색 방식의 차이

- 일반 Dense Retriever: 질문 전체를 embedding한 뒤 의미가 가까운 문서를 찾는다. metadata 조건을 자동으로 만들지 않는다.
- Metadata Filtering: 개발자가 `{'category': {'$in': ['AI']}}` 같은 filter 딕셔너리를 직접 작성한다.
- Self-query Retriever: 사용자의 자연어 조건을 LLM이 `StructuredQuery`로 바꾸고 filter를 자동 생성한다.

### Self-query 처리 흐름

`자연어 질문 → query_constructor → StructuredQuery → translator → Pinecone 검색 → list[Document]`

- `query_constructor`: 자연어 질문을 검색어·filter·limit으로 분리한다.
- `StructuredQuery`: 분리한 검색 조건을 담는 LangChain 객체이다.
- `translator`: LangChain의 공통 filter 표현을 Pinecone 문법으로 변환한다.
- Pinecone: 변환된 검색어와 filter를 적용해 문서를 반환한다.

먼저 `04_metadata_filtering.ipynb`의 upsert까지 실행해 `adv-rag-meta`에 본문과 `author`, `category` metadata를 저장해야 한다. 이 노트북은 같은 인덱스를 검색한다.


## Self-query Retriever 실행 패키지 준비

`%pip`은 현재 Jupyter 커널에 패키지를 설치한다. 설치 뒤 커널이 이전 모듈을 계속 사용하면 한 번 재시작한다.

- `langchain-classic`: 호환 기능으로 유지되는 `SelfQueryRetriever`와 query constructor를 제공한다.
- `langchain-core`: StructuredQuery와 translator의 공통 인터페이스를 제공한다.
- `lark`: LLM이 만든 filter 표현을 LangChain 객체로 파싱한다.
- `langchain-openai`: Chat Model과 embedding 모델을 연결한다.
- `langchain-pinecone`, `pinecone`: Pinecone Vector Store에 연결한다.
- `pandas`, `numpy`: 질의 데이터와 검색 평가지표를 처리한다.
- `python-dotenv`, `gdown`, `tqdm`: 환경 변수 로드, 데이터 다운로드, 진행률 표시에 사용한다.

Pinecone filter translator는 `langchain-core`의 `Visitor` 인터페이스로 직접 구현한다. 유지보수가 종료되는 `langchain-community` translator에 의존하지 않으면서 변환 원리도 확인할 수 있다.


In [1]:
%pip install -U pandas numpy langchain langchain-core langchain-classic langchain-openai langchain-pinecone pinecone lark python-dotenv gdown tqdm

  Using cached pinecone-9.1.0-cp310-abi3-win_amd64.whl.metadata (6.3 kB)
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## `.env`와 모델 설정 준비

`.env`의 OpenAI와 Pinecone API key를 환경 변수로 불러온다. 비밀값은 화면에 출력하지 않는다.

- `OPENAI_CHAT_MODEL`: 자연어를 StructuredQuery로 바꿀 Chat Model 이름이다.
- `OPENAI_EMBEDDING_MODEL`: 검색어를 vector로 바꿀 embedding 모델 이름이다.
- `PINECONE_INDEX_DIMENSION`: 저장된 vector와 새 query vector가 가져야 하는 차원 수이다.
- 기본값: 각각 `gpt-5.6-luna`, `text-embedding-3-small`, `1536`을 사용한다.


In [2]:
import os
from dotenv import find_dotenv, load_dotenv

load_dotenv(dotenv_path=find_dotenv(usecwd=True), override=False)

OPENAI_LLM_MODEL = os.getenv('OPENAI_CHAT_MODEL', 'gpt-5.6-luna')
OPENAI_EMBEDDING_MODEL = os.getenv(
    'OPENAI_EMBEDDING_MODEL',
    'text-embedding-3-small',
)
PINECONE_INDEX_DIMENSION = int(os.getenv('PINECONE_INDEX_DIMENSION', '1536'))


## 메타데이터 질의 데이터 다운로드

`queries_meta_v2.csv`에는 자연어 질문과 질의별 정답 문서가 함께 들어 있다.

- `query_id`: 질문을 구분하는 ID이다.
- `query_text`: 저자·카테고리 조건이 포함된 자연어 질문이다.
- `relevant_doc_ids`: 해당 질문의 정답 문서 ID와 관련성 등급을 저장한 qrels 문자열이다.
- qrels(Query Relevance Judgments): 검색 결과를 평가할 때 사용하는 질의별 정답표이다.

`query_text`는 query constructor의 입력으로 사용한다. `relevant_doc_ids`는 Dense와 Self-query가 정답 문서를 얼마나 잘 찾았는지 비교할 때 사용한다.


In [3]:
!gdown 1AQ_a5M9hp7fEc-5LTemM24Zacs1jqCwW

Downloading...
From: https://drive.google.com/uc?id=1AQ_a5M9hp7fEc-5LTemM24Zacs1jqCwW
To: C:\SKN_AI\09_llm\07_advanced_rag\01_retrieval_optimization\queries_meta_v2.csv

  0%|          | 0.00/2.38k [00:00<?, ?B/s]
100%|██████████| 2.38k/2.38k [00:00<00:00, 946kB/s]


## 자연어 조건 질의 확인

`pandas.read_csv()`는 CSV를 행과 열이 있는 `DataFrame`으로 읽는다. 표시된 표에서 다음 세 열을 확인한다.

- `query_id`: 뒤의 결과 딕셔너리와 qrels를 연결하는 key이다.
- `query_text`: Self-query Retriever에 전달할 자연어 입력이다.
- `relevant_doc_ids`: 뒤의 평가 함수가 사용할 정답 문서 문자열이다.


In [4]:
import pandas as pd

queries_df = pd.read_csv("queries_meta_v2.csv")
queries_df.head(3)

,query_id,query_text,relevant_doc_ids
0,Q01,저자 김민수의 문서를 모두 보여줘,D1=1;D11=1;D21=1
1,Q02,저자 이영희의 문서를 모두 보여줘,D2=1;D12=1;D22=1
2,Q03,저자 박지훈의 문서를 모두 보여줘,D3=1;D13=1;D23=1


## `04_metadata_filtering.ipynb`의 Vector Store 연결

앞 실습에서 본문 vector와 `doc_id`, `author`, `category` metadata를 Pinecone의 `adv-rag-meta` index에 저장했다. 이 셀은 새 문서를 저장하지 않고 기존 index를 검색할 `PineconeVectorStore` 객체를 만든다.

- index: vector와 metadata를 검색할 수 있게 모아 둔 Pinecone의 저장 단위이다.
- upsert: 같은 ID가 없으면 추가하고 있으면 갱신하는 저장 작업이다.
- `OpenAIEmbeddings`: query 문자열을 저장된 문서와 같은 차원의 vector로 변환한다.
- `PineconeVectorStore`: LangChain의 검색 요청을 Pinecone index에 전달한다.
- `index_name`: 연결할 Pinecone index 이름이다.
- `embedding`: 질의를 vector로 바꿀 embedding 객체이다.
- dimension: vector를 구성하는 숫자의 개수이며 index 생성 시 설정과 같아야 한다.
- cosine similarity: 두 vector의 방향이 가까운 정도이며 값이 클수록 의미가 가깝다고 해석한다.


In [5]:
from langchain_openai import OpenAIEmbeddings
from langchain_pinecone import PineconeVectorStore

PINECONE_META_INDEX_NAME = 'adv-rag-meta'

embeddings = OpenAIEmbeddings(
    model=OPENAI_EMBEDDING_MODEL,
    dimensions=PINECONE_INDEX_DIMENSION
)

vector_store = PineconeVectorStore(
    index_name=PINECONE_META_INDEX_NAME,
    embedding=embeddings,
)

## Self-query Retriever 구성 1: Pinecone filter translator

Self-query Retriever는 자연어를 Pinecone에 바로 전달하지 않고 다음 세 단계를 연결한다.

1. `query_constructor`: 자연어를 LangChain의 `StructuredQuery`로 변환한다.
2. `translator`: StructuredQuery의 조건을 Pinecone filter 딕셔너리로 변환한다.
3. `Vector Store`: 의미 검색어와 filter를 이용해 `list[Document]`를 반환한다.

이 코드셀은 두 번째 단계인 `PineconeFilterTranslator`를 정의한다. metadata 스키마와 최종 Retriever 객체는 뒤의 연결 셀에서 만든다.

### translator가 필요한 이유

LangChain과 Pinecone은 같은 검색 조건을 서로 다른 형식으로 표현한다. translator는 LangChain 형식을 Pinecone이 읽을 수 있는 형식으로 바꾸는 중간 변환기이다.

```text
자연어: 김민수가 작성한 문서를 찾아줘
→ StructuredQuery: author EQ 김민수
→ Pinecone filter: {'filter': {'author': {'$eq': '김민수'}}}
```

### 코드에서 정의하는 메서드

- `visit_comparison()`: `author EQ 김민수` 같은 조건 하나를 변환한다.
- `visit_operation()`: 여러 조건을 `$and` 또는 `$or`로 묶는다.
- `visit_structured_query()`: 의미 검색 문자열과 Pinecone 검색 옵션을 tuple로 반환한다.
- `accept(self)`: 조건 종류에 맞는 `visit_...()` 메서드를 자동으로 선택한다.


In [ ]:
from langchain_core.structured_query import (
    Comparator,
    Comparison,
    Operation,
    Operator,
    StructuredQuery,
    Visitor,
)

## StructuredQuery 생성용 Chat Model 준비

`ChatOpenAI`는 사용자에게 답변하기 위해서가 아니라 자연어를 StructuredQuery로 변환하기 위해 사용한다. 객체 생성만으로는 API를 호출하지 않으며, 뒤에서 query constructor를 `invoke()`할 때 요청이 발생한다.

- `model`: 사용할 Chat Model 이름이다.
- `use_responses_api=True`: Responses API로 모델을 호출한다.
- `temperature=0`: query 구조의 불필요한 변화를 줄인다.
- `reasoning_effort='none'`: 별도 추론 토큰 없이 짧은 구조 변환을 수행한다.


In [ ]:
from langchain_openai import ChatOpenAI

## Self-query Retriever 구성 2: metadata 스키마와 객체 연결

`AttributeInfo` 두 개로 LLM이 사용할 metadata 스키마를 만든다. 그런 다음 LLM, Vector Store와 앞에서 정의한 translator를 `SelfQueryRetriever.from_llm()`으로 연결한다.

- `author`: 저자 한 명을 저장하므로 `string`이다.
- `category`: 문서 하나가 여러 범주를 가질 수 있으므로 `list[string]`이다.
- 기본 반환 개수: `search_kwargs={'k': 5}`를 사용한다.
- 수량 요청: `enable_limit=True`이므로 자연어 limit이 있으면 기본 k를 덮어쓴다.


In [ ]:
from langchain_classic.chains.query_constructor.schema import AttributeInfo
from langchain_classic.retrievers.self_query.base import SelfQueryRetriever

## 자연어 저자 조건으로 문서 검색

`invoke()`에 자연어 질문 하나를 전달하면 Self-query의 전체 흐름이 실행된다.

`자연어 질문 → StructuredQuery 생성 → Pinecone filter 변환 → vector 검색 → list[Document]`

- `Document`: 검색된 문서 하나를 나타내는 LangChain 객체이다.
- `page_content`: 검색된 본문 문자열이다.
- `metadata`: `doc_id`, `author`, `category` 같은 부가정보 딕셔너리이다.
- `list[Document]`: 관련성이 높은 순서로 반환된 Document 목록이며 LLM 답변을 생성하지 않는다.

실행 결과의 모든 `author`가 질문의 `김민수`와 일치하는지 확인한다.


## 자연어 질문이 Pinecone 검색 조건으로 바뀌는 과정

이번에는 Self-query Retriever의 내부 두 단계를 따로 확인한다. 이 셀에서는 Pinecone 검색을 실행하지 않는다.

1. `query_constructor`: 자연어를 `StructuredQuery(query, filter, limit)`으로 변환한다.
2. `translator`: StructuredQuery의 filter를 Pinecone filter 딕셔너리로 변환한다.

`김민수가 작성한 문서 2개`라는 질문은 다음 구조가 되는지 확인한다.

- `query`: 본문 주제가 없으므로 빈 문자열에 가까운 값이다.
- `filter`: `Comparison(Comparator.EQ, 'author', '김민수')` 형태이다.
- `limit`: `enable_limit=True`이므로 `2`가 될 수 있다.
- Pinecone 조건: `{'filter': {'author': {'$eq': '김민수'}}, 'k': 2}` 형태이다.

LLM 출력은 모델에 따라 표현이 달라질 수 있으므로 실제 필드값을 직접 확인한다.


## OR로 연결된 카테고리 조건 확인

하나의 조건은 `Comparison`으로 표현하지만, `AI이거나 건강`처럼 여러 조건을 결합하면 `Operation`이 필요하다.

- 첫 번째 Comparison: `category == AI`이다.
- 두 번째 Comparison: `category == 건강`이다.
- Operation: 두 Comparison을 `Operator.OR`로 묶는다.
- translator 결과: Pinecone의 `$or` 안에 두 `$eq` 조건이 들어간다.

같은 질문으로 실제 검색까지 실행하고 반환 문서의 `category`에 `AI` 또는 `건강`이 포함되는지 확인한다.


## 검색 결과 평가를 위한 qrels 변환

Self-query가 filter를 자동 생성하는 것만으로 검색 품질이 좋아졌다고 판단할 수는 없다. Dense와 Self-query의 검색 문서 ID를 질의별 정답표인 qrels와 비교해야 한다.

`검색 결과 문서 ID → qrels 정답과 비교 → 질의별 지표 → 전체 평균`

- qrels 입력: `D6=3;D14=2` 같은 문자열이다.
- 변환 결과: `{'D6': 3, 'D14': 2}` 같은 딕셔너리이다.
- key: 정답 문서 ID이다.
- value: 관련성 등급이며 이 실습에서는 1 이상을 정답으로 처리한다.

문자열을 딕셔너리로 바꾸면 검색된 `doc_id`가 정답 key에 포함되는지 정확히 검사할 수 있다.


## 질의 하나의 검색 지표 계산

검색 결과는 관련 문서를 **찾았는지**뿐 아니라 **몇 위에서 찾았는지**도 평가해야 한다. `k=5`는 최대 반환 개수가 아니라 평가할 상위 순위의 범위인 cutoff이다. 결과가 5개보다 적으면 비어 있는 순위는 관련 없는 결과로 처리되어 P@5가 낮아진다.

- P@k(Precision at k): 상위 k개 중 정답 문서의 비율이다.
- R@k(Recall at k): 전체 정답 문서 중 상위 k개에서 찾은 비율이다.
- RR(Reciprocal Rank): 처음 등장한 정답 문서 순위의 역수이다. 1위이면 1.0, 2위이면 0.5이다.
- AP@k(Average Precision at k): 정답을 만난 각 순위의 Precision을 평균해 정답의 순서와 누락을 함께 반영한다.

함수는 네 값을 `(precision, recall, rr, ap)` tuple로 반환한다.


## 모든 질의의 평균 검색 지표 계산

`evaluate_all()`은 Q01부터 Q30까지 같은 평가를 반복하고 질의별 지표의 평균을 계산한다.

- `method_results`: `query_id → 순서가 있는 문서 ID 목록` 딕셔너리이다.
- `queries_df`: 각 query_id의 자연어 질문과 qrels를 가진 DataFrame이다.
- `per_query_metrics`: 질의마다 `(P@k, R@k, RR, AP@k)`를 저장한 목록이다.
- `metric_array`: `(질의 수, 4)` shape의 NumPy 배열이다. 행은 질의, 열은 네 지표이다.
- 반환값: 전체 질의의 P@k·R@k·MRR·MAP 평균 딕셔너리이다.

MRR은 질의별 RR의 평균이고, MAP는 질의별 AP의 평균이다.


In [ ]:
import numpy as np

## Dense 검색과 Self-query 검색 결과 수집

Self-query가 metadata 조건을 자동 생성하는 것만으로 검색 성능이 좋아졌다고 판단할 수는 없다. 같은 30개 질문을 두 방식으로 검색한 뒤, 반환된 문서 ID를 qrels 정답과 비교한다.

- Dense 기준선: metadata filter 없이 질문 전체를 embedding해 검색한다.
- Self-query: 질문을 의미 검색어와 metadata 조건으로 분리해 검색한다.
- 수집 결과: `query_id → 검색된 doc_id 목록`으로 저장한다.

두 결과의 key와 반환 개수를 같게 만들어 앞에서 정의한 `evaluate_all()`에 전달한다.


In [ ]:
from tqdm import tqdm

## Self-query 결과 딕셔너리 확인

`self_query_results`가 평가 함수의 입력 형식과 맞는지 확인한다.

- key: Q01부터 Q30까지의 `query_id`이다.
- value: Self-query가 반환한 순서가 유지된 `doc_id` 목록이다.
- 확인 기준: 30개 key가 있고, 각 문서 ID가 `D숫자` 형태인지 확인한다.

결과가 예상과 다르면 문서 ID만 보지 말고 앞에서 출력한 StructuredQuery와 실제 metadata 타입을 함께 점검한다.


## Dense와 Self-query 검색 지표 비교

두 검색 결과를 같은 qrels와 `k=5`로 평가해 비교표를 만든다.

- `Dense`: metadata filter 없이 질문 전체를 embedding한 기준선이다.
- `Self-query`: LLM이 semantic query와 metadata filter를 분리한 결과이다.
- `Metric`: P@5, R@5, MRR, MAP의 이름이다.

Self-query가 P@5를 높이더라도 관련 문서를 지나치게 제외하면 R@5가 낮아질 수 있다. 하나의 지표만으로 개선을 판단하지 않는다.
